In [5]:
! pip install torch torchvision opencv-python pillow

In [9]:
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# 1. Model Setup (Same as your training)
def get_model(num_classes):
    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = [
    'CP77-400', 'CPF-237', 'CPF-246', 'CPF-247', 'CPF-248', 'CPF-249', 'CPF-250',
    'CPF-251', 'CPF-252', 'CPF-253', 'CoJ-84', 'HSF-240', 'NSG-59', 'SPF-213',
    'SPF-234', 'SPF-93', 'YTFG-236'
]

model = get_model(len(class_names))
# Update the path to where your file is located
model.load_state_dict(torch.load('sugarcane_model.pth', map_location=device))
model.to(device).eval()

# 2. Preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. Webcam Loop with Logic
cap = cv2.VideoCapture(0)
THRESHOLD = 0.75  # 75% confidence required to show name

while True:
    ret, frame = cap.read()
    if not ret: break

    # Process frame
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    input_tensor = preprocess(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        confidence, idx = torch.max(probs, 0)

    # --- LOGIC: Only show name if confidence is high ---
    if confidence.item() > THRESHOLD:
        label = f"{class_names[idx]} ({confidence.item()*100:.1f}%)"
        color = (0, 255, 0) # Green
    else:
        label = "Searching for Sugarcane..."
        color = (0, 0, 255) # Red

    cv2.putText(frame, label, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2.imshow('Sugarcane Detector', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()